In [2]:
from pathlib import Path
from stable_baselines3 import SAC, DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import (
    DummyVecEnv, VecMonitor, VecNormalize, VecFrameStack, VecCheckNan, VecTransposeImage
)
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList

SEED = 42
run_dir = Path("experiments/dqn_carracing")
run_dir.mkdir(parents=True, exist_ok=True)

env_kwargs = {"continuous": False}

# Train env
train_env = make_vec_env(
    "CarRacing-v3",
    n_envs=4,
    seed=SEED,
    vec_env_cls=DummyVecEnv,
    monitor_dir=str(run_dir / "monitor_train"),
    env_kwargs=env_kwargs
)
train_env = VecMonitor(train_env)
train_env = VecFrameStack(train_env, n_stack=4)
train_env = VecTransposeImage(train_env)
train_env = VecNormalize(train_env, norm_obs=False, norm_reward=True, clip_reward=10.0)

# Eval env (separate)
eval_env = make_vec_env(
    "CarRacing-v3",
    n_envs=1,
    seed=SEED + 1000,
    vec_env_cls=DummyVecEnv,
    monitor_dir=str(run_dir / "monitor_eval"),
    env_kwargs=env_kwargs
)
eval_env = VecMonitor(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)
eval_env = VecTransposeImage(eval_env)
eval_env = VecNormalize(eval_env, norm_obs=False, norm_reward=False, training=False)

eval_cb = EvalCallback(
    eval_env=eval_env,
    best_model_save_path=str(run_dir / "best_model"),
    log_path=str(run_dir / "eval_logs"),
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=True,
)

ckpt_cb = CheckpointCallback(
    save_freq=25_000,
    save_path=str(run_dir / "checkpoints"),
    name_prefix="sac",
)

model = DQN(
    "CnnPolicy",
    train_env,
    seed=SEED,
    device="mps",
    verbose=1,
    tensorboard_log=str(run_dir / "tb"),
)

model.learn(total_timesteps=100_000, callback=CallbackList([eval_cb, ckpt_cb]))
model.save(str(run_dir / "final_model"))
train_env.save(str(run_dir / "vecnormalize.pkl"))

/Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(
/Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 221.20GB > 11.34GB
  warnings.warn(


Using mps device
Logging to experiments/dqn_carracing/tb/DQN_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1e+03    |
|    ep_rew_mean      | -45      |
|    exploration_rate | 0.62     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 228      |
|    time_elapsed     | 17       |
|    total_timesteps  | 4000     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00062  |
|    n_updates        | 243      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 998      |
|    ep_rew_mean      | -45.5    |
|    exploration_rate | 0.24     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 242      |
|    time_elapsed     | 33       |
|    total_timesteps  | 8000     |
| train/              |          |
|    learning_rate    | 0.0

In [6]:
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import (
    DummyVecEnv, VecMonitor, VecNormalize, VecFrameStack, VecTransposeImage
)

run_dir = Path("experiments/dqn_carracing")
env_kwargs = {"continuous": False}

# Build the SAME pre-normalize wrapper stack as training
eval_env = make_vec_env(
    "CarRacing-v3",
    n_envs=1,
    vec_env_cls=DummyVecEnv,
    env_kwargs=env_kwargs,
)
eval_env = VecMonitor(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)
eval_env = VecTransposeImage(eval_env)

# Now load normalization stats (shapes match)
eval_env = VecNormalize.load(str(run_dir / "vecnormalize.pkl"), eval_env)
eval_env.training = False
eval_env.norm_reward = False

model = DQN.load(str(run_dir / "best_model" / "best_model.zip"), env=eval_env, device="mps")
mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(mean_r, std_r)

/Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(
/Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 221.20GB > 7.66GB
  warnings.warn(


121.95064 79.43959


In [11]:
vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render("human")